## Análise preliminar e tratamento dos dados

Este notebook realiza a análise preliminar e o tratamento dos dados, garantindo a integridade e consistência das informações antes de prosseguir com análises mais aprofundadas.

Arquivo de entrada: `BPS_20_26_OrlandoCastro.csv`

Arquivo de saída: `BPS_20_26_OrlandoCastro_atualizado.csv`


## 1. Verificações iniciais

In [41]:
# Importações necessárias
import pandas as pd
import os

# Configurações
pasta_dados = 'Dados'
arquivo_entrada = os.path.join(pasta_dados, 'BPS_20_26_OrlandoCastro.csv')

# Verificação de integridade do arquivo final e criação do dataframe para análise preliminar
if os.path.exists(arquivo_entrada):
    df_verificacao = pd.read_csv(arquivo_entrada, sep=';', encoding='utf-8', dtype=str)
    print(f"\nArquivo de dados consolidado inicial '{arquivo_entrada}' contém {len(df_verificacao)} linhas e {len(df_verificacao.columns)} colunas.")
    


Arquivo de dados consolidado inicial 'Dados\BPS_20_26_OrlandoCastro.csv' contém 342716 linhas e 25 colunas.


In [42]:
# Análise preliminar dos dados consolidados
# Exibe informações gerais sobre o dataframe
print("\nInformações gerais do arquivo consolidado:")   
print(df_verificacao.info())



Informações gerais do arquivo consolidado:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 342716 entries, 0 to 342715
Data columns (total 25 columns):
 #   Column                           Non-Null Count   Dtype 
---  ------                           --------------   ----- 
 0   ano_compra                       342716 non-null  object
 1   nome_instituicao                 342559 non-null  object
 2   esfera                           342716 non-null  object
 3   cnpj_instituicao                 342716 non-null  object
 4   municipio_instituicao            342716 non-null  object
 5   uf                               342716 non-null  object
 6   compra                           342716 non-null  object
 7   insercao                         340588 non-null  object
 8   codigo_br                        342716 non-null  object
 9   descricao_catmat                 342716 non-null  object
 10  unidade_fornecimento             342691 non-null  object
 11  generico                         1

In [43]:
# Exibe as primeiras linhas do dataframe para uma visão geral
print("\nVisualização das primeiras linhas do arquivo consolidado:")
print(df_verificacao.head())



Visualização das primeiras linhas do arquivo consolidado:
  ano_compra                                   nome_instituicao     esfera  \
0       2020             FUNDO  MUNICIPAL  DE  SAUDE DE  MARABA  MUNICIPAL   
1       2020  FUNDO MUNICIPAL DE SAUDE - MUNICIPIO DE ALTO P...  MUNICIPAL   
2       2020                   MUNICIPIO DE AMERICO BRASILIENSE  MUNICIPAL   
3       2020                   MUNICIPIO DE AMERICO BRASILIENSE  MUNICIPAL   
4       2020                   MUNICIPIO DE AMERICO BRASILIENSE  MUNICIPAL   

     cnpj_instituicao municipio_instituicao  uf      compra    insercao  \
0  18.478.187/0001-07                MARABA  PA  01/01/2020  19/01/2024   
1  08.533.932/0001-01           ALTO PARANA  PR  01/01/2020  13/03/2020   
2  43.976.166/0001-50   AMERICO BRASILIENSE  SP  01/01/2020  06/10/2020   
3  43.976.166/0001-50   AMERICO BRASILIENSE  SP  01/01/2020  06/10/2020   
4  43.976.166/0001-50   AMERICO BRASILIENSE  SP  01/01/2020  06/10/2020   

  codigo_br          

In [44]:
# Verifica se há duplicatas no dataframe
print("\nVerificação de duplicatas no arquivo consolidado:")
print(df_verificacao.duplicated().sum())

# Exibe linhas de dados duplicadas, se houver
duplicatas = df_verificacao[df_verificacao.duplicated(keep=False)]
if not duplicatas.empty:
    print("\nLinhas duplicadas encontradas:")
    print(duplicatas)


Verificação de duplicatas no arquivo consolidado:
19

Linhas duplicadas encontradas:
       ano_compra                                   nome_instituicao  \
290774       2024                            MUNICIPIO DO RIO GRANDE   
290790       2024                            MUNICIPIO DO RIO GRANDE   
290793       2024  FUNDO MUNICIPAL DE SAUDE DE ESPIGAO DO OESTE (...   
290824       2024  FUNDO MUNICIPAL DE SAUDE DE ESPIGAO DO OESTE (...   
295407       2024                      SECRETARIA DE ESTADO DA SAUDE   
295411       2024                      SECRETARIA DE ESTADO DA SAUDE   
295412       2024                      SECRETARIA DE ESTADO DA SAUDE   
295414       2024                      SECRETARIA DE ESTADO DA SAUDE   
296760       2024   FUNDO MUNICIPAL DE SAUDE DE ITAPUA DO OESTE - RO   
296796       2024   FUNDO MUNICIPAL DE SAUDE DE ITAPUA DO OESTE - RO   
296820       2024   FUNDO MUNICIPAL DE SAUDE DE ITAPUA DO OESTE - RO   
296837       2024   FUNDO MUNICIPAL DE SAUDE DE IT

In [45]:
# Elimina duplicatas, mantendo a primeira ocorrência
df_verificacao_sem_duplicatas = df_verificacao.drop_duplicates(keep='first')
print(f"\nApós remover duplicatas, o arquivo consolidado contém {len(df_verificacao_sem_duplicatas)} linhas.")


Após remover duplicatas, o arquivo consolidado contém 342697 linhas.


In [46]:
# Verifica se há valores nulos no dataframe
print("\nVerificação de valores nulos no arquivo consolidado:")
print(df_verificacao.isnull().sum())



Verificação de valores nulos no arquivo consolidado:
ano_compra                              0
nome_instituicao                      157
esfera                                  0
cnpj_instituicao                        0
municipio_instituicao                   0
uf                                      0
compra                                  0
insercao                             2128
codigo_br                               0
descricao_catmat                        0
unidade_fornecimento                   25
generico                           172037
anvisa                             172037
modalidade_compra                       0
tipo_compra                             0
capacidade                         218035
unidade_medida                     218035
unidade_fornecimento_capacidade        25
cnpj_fornecedor                         0
fornecedor                              0
cnpj_fabricante                         0
fabricante                              0
qtd_itens_comprados   

## 2. Tratamento de dados nulos

### 2.1. Capacidades e Unidades de Medida

Os registros das compras na base de dados do BPS (Banco de Preços em Saúde) são organizados em unidades de fornecimento, que representam a forma como os produtos ou serviços são entregues. Cada unidade de fornecimento possui uma capacidade específica, que indica a quantidade ou volume do produto ou serviço fornecido. Além disso, cada unidade de fornecimento está associada a uma unidade de medida, que define a forma como a capacidade é expressa.

Uma mesma unidade de fornecimento pode ter diferentes capacidades e unidades de medida, dependendo do contexto ou da forma como o produto ou serviço é oferecido. Por exemplo, um medicamento pode ser fornecido em comprimidos, cápsulas ou frascos, cada um com uma capacidade diferente e uma unidade de medida específica.

No entanto, há casos em que o valor do campo `unidade_fornecimento_capacidade` é igual ao do campo `unidade_fornecimento`, isso indica que a capacidade da unidade de fornecimento é a mesma que a própria unidade de fornecimento. Por exemplo, se a `unidade_fornecimento` for "comprimido" e a `unidade_fornecimento_capacidade` também for "comprimido", isso significa que a capacidade da unidade de fornecimento é de um comprimido. Nesses casos, a unidade de medida associada à capacidade será a própria unidade de medida que a representa, como "unidade" ou "peça", dependendo do contexto do produto ou serviço fornecido.

Dessa forma, para fins das análises, é possível atribuir `capacidade` igual a 1 e `unidade_medida` igual a `unidade_fornecimento` para os registros em que a capacidade da unidade de fornecimento é a mesma que a própria unidade de fornecimento. Isso permite padronizar os dados e facilitar a comparação entre diferentes unidades de fornecimento, capacidades e unidades de medida.

In [47]:
# Atribuir o valor 1 à capacidade e "unidade" à unidade de medida para registros onde a capacidade é igual à unidade de fornecimento
condicao = df_verificacao['unidade_fornecimento_capacidade'] == df_verificacao['unidade_fornecimento']
df_verificacao.loc[condicao, 'capacidade'] = float(1.0)
df_verificacao.loc[condicao, 'unidade_medida'] = df_verificacao['unidade_fornecimento']
print("\nAjustes realizados para registros onde a capacidade é igual à unidade de fornecimento:")
print(df_verificacao[condicao][['unidade_fornecimento', 'unidade_fornecimento_capacidade', 'unidade_medida']])



Ajustes realizados para registros onde a capacidade é igual à unidade de fornecimento:
       unidade_fornecimento unidade_fornecimento_capacidade unidade_medida
1                   UNIDADE                         UNIDADE        UNIDADE
2                COMPRIMIDO                      COMPRIMIDO     COMPRIMIDO
3                COMPRIMIDO                      COMPRIMIDO     COMPRIMIDO
4                COMPRIMIDO                      COMPRIMIDO     COMPRIMIDO
5                   UNIDADE                         UNIDADE        UNIDADE
...                     ...                             ...            ...
342708        FRASCO-AMPOLA                   FRASCO-AMPOLA  FRASCO-AMPOLA
342709           COMPRIMIDO                      COMPRIMIDO     COMPRIMIDO
342711           COMPRIMIDO                      COMPRIMIDO     COMPRIMIDO
342712        FRASCO-AMPOLA                   FRASCO-AMPOLA  FRASCO-AMPOLA
342714           COMPRIMIDO                      COMPRIMIDO     COMPRIMIDO

[218035 row

### 2.2. Campos `unidade_fornecimento` e `unidade_fornecimento_capacidade` nulos

Há 25 registros com `unidade_fornecimento` e `unidade_fornecimento_capacidade` nulos. No entanto, após exame dos dados, verificou-se que todos esses registros possuem os valores do campo `unidade_medida` preenchidos e todos são itens comercilizados por unidade. Dessa forma, com vistas a manter a padronização definida no item 1, atribuímos `unidade_fornecimento` e `unidade_fornecimento_capacidade` com o valor "unidade" para esses registros.

In [48]:
# Atribuir o valor "unidade" aos campos `unidade_fornecimento` e `unidade_fornecimento_capacidade` para regsitros onde estes campos estão nulos
condicao_nulos = df_verificacao['unidade_fornecimento'].isnull() | df_verificacao['unidade_fornecimento_capacidade'].isnull()
df_verificacao.loc[condicao_nulos, 'unidade_fornecimento'] = 'unidade'
df_verificacao.loc[condicao_nulos, 'unidade_fornecimento_capacidade'] = 'unidade'
print("\nAjustes realizados para registros onde os campos `unidade_fornecimento` ou `unidade_fornecimento_capacidade` estão nulos:")
print(df_verificacao[condicao_nulos][['unidade_fornecimento', 'unidade_fornecimento_capacidade']])  



Ajustes realizados para registros onde os campos `unidade_fornecimento` ou `unidade_fornecimento_capacidade` estão nulos:
       unidade_fornecimento unidade_fornecimento_capacidade
1247                unidade                         unidade
19572               unidade                         unidade
28747               unidade                         unidade
28749               unidade                         unidade
52831               unidade                         unidade
52899               unidade                         unidade
89375               unidade                         unidade
92682               unidade                         unidade
111315              unidade                         unidade
161305              unidade                         unidade
200372              unidade                         unidade
211188              unidade                         unidade
214721              unidade                         unidade
236805              unidade          

### 2.3. Data da inserção do registro no BPS

Os campos `compra` e o campo `insercao` representam as data em que uma compra foi realizada e a data da inserção dos dados dessa compra no sistema BPS, respectivamente. 

Há 2.128 registros com o campo `insercao` nulo. No entanto, todos os registros possuem o campo `compra` preenchido. Portanto, para fins de análise, é possível utilizar o campo `compra` como referência temporal, mesmo nos casos em que a data de inserção do registro está ausente.

Com vistas a evitar inconsistências e manter a padronização, optou-se por remover o campo `insercao` do arquivo consolidado, mantendo apenas o campo `compra` para referência temporal.


In [49]:
# Remove as as colunas `insercao` do dataframe
print("\nRemovendo a coluna 'insercao' do dataframe...")
# verifica se as colunas existem antes de tentar removê-las
if 'insercao' in df_verificacao.columns:
    df_verificacao.drop(columns=['insercao'], inplace=True)
print("Colunas 'insercao' removida com sucesso.")


Removendo a coluna 'insercao' do dataframe...
Colunas 'insercao' removida com sucesso.


### 2.4. Registros com campos `generico` e `anvisa` nulos

O campo `generico` indica se o item é um medicamento genérico, conforme a regulamentação da Agência Nacional de Vigilância Sanitária (Anvisa).
O campo `anvisa` indica o código CMED que é número de registro na Anvisa, que certifica a autorização para comercialização e uso do produto no Brasil.

Há 172.037 registros (50,2% do total) com os campos `generico` e `anvisa` nulos. No entanto, todos os registros possuem o campo `descricao_catmat` preenchido, o que permite identificar o item adquirido. Portanto, para fins de análise, é possível utilizar o campo `descricao_catmat` como referência para identificar os itens, mesmo nos casos em que os campos `generico` e `anvisa` estão ausentes.

A grande quantidade de registros com campos `generico` e `anvisa` nulos os tornam pouco uteis para análises. Assim, e de modo a reduzir o tamanho do arquivo e facilitar a análise, optou-se por remover os campos `generico` e `anvisa` do arquivo consolidado, mantendo apenas o campo `descricao_catmat` para identificar os itens adquiridos.

In [50]:
# Remove as as colunas `generico` e `anvisa` do dataframe
print("\nRemovendo as colunas 'generico' e 'anvisa' do arquivo consolidado...")
# verifica se as colunas existem antes de tentar removê-las
if 'generico' in df_verificacao.columns and 'anvisa' in df_verificacao.columns:
    df_verificacao.drop(columns=['generico', 'anvisa'], inplace=True)
print("Colunas removidas com sucesso.")


Removendo as colunas 'generico' e 'anvisa' do arquivo consolidado...
Colunas removidas com sucesso.


### 2.5. Nomes de instituições nulos

Há 157 registros com o campo `nome_instituicao` nulo. Esses registros correspondem a compras realizadas por órgãos ou entidades que não possuem um nome de instituição associado. No entanto, todos os registros possuem o campo `cnpj_instituicao` preenchido, o que permite identificar a instituição responsável pela compra. Portanto, para fins de análise, é possível utilizar o campo `cnpj_instituicao` como identificador da instituição, mesmo nos casos em que o nome da instituição está ausente.

De modo a recuperar os nomes das instituições nulas, utilizamos o campo `cnpj_instituicao` para buscar os nomes correspondentes em uma base de dados externa, por meio de consulta à API do [OpenCNPJ.org](https://api.opencnpj.org). Dessa forma, foi possível preencher os registros com os nomes corretos das instituições, garantindo a integridade e a completude dos dados para análise.

Como resultado, temos o arquivo `BPS_26_OrlandoCastro_atualizado.csv`.

In [51]:
# Importações necessárias
import requests
import time
import pandas as pd

ARQUIVO_SAIDA = os.path.join(pasta_dados, "BPS_20_26_OrlandoCastro_atualizado.csv")

def buscar_razao_social(cnpj):
    url = f"https://api.opencnpj.org/{cnpj}"
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    dados = resp.json()
    return dados.get("razao_social")

mask_nome_vazio = df_verificacao["nome_instituicao"].isna() | (df_verificacao["nome_instituicao"].astype(str).str.strip() == "")
mask_cnpj_preenchido = df_verificacao["cnpj_instituicao"].notna() & (df_verificacao["cnpj_instituicao"].astype(str).str.strip() != "")

total_sem_nome = int(mask_nome_vazio.sum())

cnpjs_unicos = df_verificacao.loc[mask_nome_vazio & mask_cnpj_preenchido, "cnpj_instituicao"].astype(str).str.strip().unique().tolist()

razoes_sociais = {}
nao_encontrados = []

for cnpj in cnpjs_unicos:
    try:
        razao = buscar_razao_social(cnpj)
        if razao:
            razoes_sociais[cnpj] = razao
        else:
            nao_encontrados.append(cnpj)
    except Exception:
        nao_encontrados.append(cnpj)
    time.sleep(0.5)

for cnpj, razao in razoes_sociais.items():
    mask_update = (
        (df_verificacao["cnpj_instituicao"].astype(str).str.strip() == cnpj)
        & (df_verificacao["nome_instituicao"].isna() | (df_verificacao["nome_instituicao"].astype(str).str.strip() == ""))
    )
    df_verificacao.loc[mask_update, "nome_instituicao"] = razao

mask_ainda_vazio = df_verificacao["nome_instituicao"].isna() | (df_verificacao["nome_instituicao"].astype(str).str.strip() == "")
registros_preenchidos = total_sem_nome - int(mask_ainda_vazio.sum())

df_verificacao.to_csv(ARQUIVO_SAIDA, sep=";", index=False, encoding="utf-8")

print(f"Total de registros sem nome_instituicao: {total_sem_nome}")
print(f"Total de CNPJs unicos consultados na API: {len(cnpjs_unicos)}")
print(f"Registros preenchidos com sucesso: {registros_preenchidos}")
print(f"CNPJs nao encontrados pela API ({len(nao_encontrados)}): {nao_encontrados}")
print(f"Arquivo salvo em: {ARQUIVO_SAIDA}")


Total de registros sem nome_instituicao: 157
Total de CNPJs unicos consultados na API: 26
Registros preenchidos com sucesso: 157
CNPJs nao encontrados pela API (0): []
Arquivo salvo em: Dados\BPS_20_26_OrlandoCastro_atualizado.csv


## 3. Verificações finais e conclusão do processo de tratamento preliminar de dados

Após a realização das etapas de tratamento preliminar dos dados, foram realizadas verificações finais para garantir a integridade e a consistência dos dados no arquivo consolidado. As principais verificações incluem:
- Verificação de duplicidade de registros: Foram identificados e removidos registros duplicados, garantindo que cada compra seja representada apenas uma vez no arquivo consolidado.
- Verificação de consistência de tipos de dados: Foram verificadas as colunas do arquivo consolidado para garantir que os tipos de dados estejam corretos e consistentes com as definições originais dos arquivos CSV. Isso inclui a verificação de campos numéricos, datas e strings, garantindo que os dados estejam formatados corretamente para análise.
- Verificação de valores nulos: Foram realizadas verificações adicionais para identificar e tratar valores nulos em colunas críticas, garantindo que os dados estejam completos e consistentes para análise. Isso inclui a verificação de campos obrigatórios, como `compra`, `descricao_catmat` e `cnpj_instituicao`, garantindo que não haja registros com informações ausentes que possam comprometer a análise.


In [62]:
# Verifica estrutura do DataFrame e existência de nulos.

print("\nAnálise Estrutural do DataFrame:")

# Cria um novo DataFrame com as estatísticas das colunas
df_info_formatado = pd.DataFrame({
    'Tipo de Dado': df_verificacao.dtypes,
    'Valores Não-Nulos': df_verificacao.notnull().sum(),
    'Valores Nulos': df_verificacao.isnull().sum(),
    '% Nulos': (df_verificacao.isnull().sum() / len(df_verificacao) * 100).round(2)
})

# Exibe o resultado como uma tabela limpa
print(df_info_formatado)



Análise Estrutural do DataFrame:
                                Tipo de Dado  Valores Não-Nulos  \
ano_compra                            object             342716   
nome_instituicao                      object             342716   
esfera                                object             342716   
cnpj_instituicao                      object             342716   
municipio_instituicao                 object             342716   
uf                                    object             342716   
compra                                object             342716   
codigo_br                             object             342716   
descricao_catmat                      object             342716   
unidade_fornecimento                  object             342716   
modalidade_compra                     object             342716   
tipo_compra                           object             342716   
capacidade                            object             342716   
unidade_medida              